In [0]:
# ============================================================================
# Rearc Data Quest - Raw Data Ingestion
# ----------------------------------------------------------------------------
# Downloads the full contents of the BLS productivity time-series folder
# (https://download.bls.gov/pub/time.series/pr/) and the DataUSA population
# API, landing everything as raw files in a Databricks Volume.
#
# Idempotent: files already present in the Volume are skipped on re-runs.
# Dynamic: BLS file names are discovered by parsing the directory listing HTML,
#          so newly added/removed files at the source are handled automatically.
#
# BLS 403 resolution: per https://www.bls.gov/bls/pss.htm, BLS blocks robots
# that don't include contact information. We set a User-Agent header with a
# contact email, which complies with their data access policy.
# ============================================================================

import requests
import os
import re
import json
from datetime import datetime, timezone

# --- Configuration -----------------------------------------------------------
BLS_BASE_URL = "https://download.bls.gov/pub/time.series/pr/"
POPULATION_API_URL = (
    "https://honolulu-api.datausa.io/tesseract/data.jsonrecords"
    "?cube=acs_yg_total_population_1"
    "&drilldowns=Year%2CNation"
    "&locale=en"
    "&measures=Population"
)

# BLS requires a User-Agent with contact info per their data access policy.
USER_AGENT = "RearcDataQuest/1.0 (Databricks data engineering; contact: aalhomedey@gmail.com)"

# Unity Catalog Volume path for raw data
RAW_VOLUME_PATH = "/Volumes/rearc_quest/bronze/raw_data"
BLS_SUBDIR = "bls_pr"
POP_SUBDIR = "population"

print(f"Raw volume: {RAW_VOLUME_PATH}")
print(f"BLS subdirectory: {BLS_SUBDIR}")
print(f"Population subdirectory: {POP_SUBDIR}")

Raw volume: /Volumes/rearc_quest/bronze/raw_data
BLS subdirectory: bls_pr
Population subdirectory: population


In [0]:
# --- Helper Functions --------------------------------------------------------

def list_volume_files(path):
    """List files that already exist in a Volume directory using dbutils."""
    try:
        files = dbutils.fs.ls(path)
        return {f.name for f in files}
    except Exception:
        return set()


def download_file(url, local_path, headers):
    """Download a file from a URL and save it to a Volume path.
    Returns True if downloaded, False if skipped (already exists)."""
    filename = os.path.basename(local_path)
    existing = list_volume_files(os.path.dirname(local_path))
    
    if filename in existing:
        print(f"  SKIP  {filename} (already exists)")
        return False
    
    resp = requests.get(url, headers=headers, timeout=60)
    resp.raise_for_status()
    
    content = resp.text
    dbutils.fs.put(local_path, content, overwrite=True)
    print(f"  DOWNLOADED  {filename}  ({len(content):,} bytes)")
    return True


def discover_bls_files(base_url, headers):
    """Parse the BLS directory listing HTML to discover all available files.
    This is dynamic: if BLS adds or removes files, we pick them up automatically.
    """
    resp = requests.get(base_url, headers=headers, timeout=30)
    resp.raise_for_status()
    
    pattern = r'<A HREF="/pub/time\.series/pr/([^"]+)">'
    filenames = re.findall(pattern, resp.text)
    filenames = [f for f in filenames if not f.endswith("/") and f != ""]
    
    return filenames


print("Helper functions defined.")

Helper functions defined.


In [0]:
# --- Step 1: Discover BLS Files ----------------------------------------------

print("=" * 70)
print("Step 1: Discovering BLS files from directory listing...")
print(f"  URL: {BLS_BASE_URL}")

bls_files = discover_bls_files(BLS_BASE_URL, {"User-Agent": USER_AGENT})
print(f"  Found {len(bls_files)} files:")
for f in sorted(bls_files):
    print(f"    - {f}")
print()

Step 1: Discovering BLS files from directory listing...
  URL: https://download.bls.gov/pub/time.series/pr/
  Found 12 files:
    - pr.class
    - pr.contacts
    - pr.data.0.Current
    - pr.data.1.AllData
    - pr.duration
    - pr.footnote
    - pr.measure
    - pr.period
    - pr.seasonal
    - pr.sector
    - pr.series
    - pr.txt



In [0]:
# --- Step 2: Download BLS Files to Volume ------------------------------------

print("=" * 70)
print("Step 2: Downloading BLS files to Volume...")
print(f"  Target: {RAW_VOLUME_PATH}/{BLS_SUBDIR}/")
print()

bls_volume_dir = f"{RAW_VOLUME_PATH}/{BLS_SUBDIR}"
downloaded_count = 0
skipped_count = 0

for filename in sorted(bls_files):
    url = f"{BLS_BASE_URL}{filename}"
    local_path = f"{bls_volume_dir}/{filename}"
    
    try:
        was_downloaded = download_file(url, local_path, {"User-Agent": USER_AGENT})
        if was_downloaded:
            downloaded_count += 1
        else:
            skipped_count += 1
    except Exception as e:
        print(f"  ERROR  {filename}: {e}")

print(f"\nBLS Summary: {downloaded_count} downloaded, {skipped_count} skipped (already existed)")

Step 2: Downloading BLS files to Volume...
  Target: /Volumes/rearc_quest/bronze/raw_data/bls_pr/

  SKIP  pr.class (already exists)
  SKIP  pr.contacts (already exists)
  SKIP  pr.data.0.Current (already exists)
  SKIP  pr.data.1.AllData (already exists)
  SKIP  pr.duration (already exists)
  SKIP  pr.footnote (already exists)
  SKIP  pr.measure (already exists)
  SKIP  pr.period (already exists)
  SKIP  pr.seasonal (already exists)
  SKIP  pr.sector (already exists)
  SKIP  pr.series (already exists)
  SKIP  pr.txt (already exists)

BLS Summary: 0 downloaded, 12 skipped (already existed)


In [0]:
# --- Step 3: Download Population API Data ------------------------------------

print("=" * 70)
print("Step 3: Downloading Population API data...")
print(f"  URL: {POPULATION_API_URL}")
print()

pop_volume_dir = f"{RAW_VOLUME_PATH}/{POP_SUBDIR}"
pop_filename = "population.json"
pop_local_path = f"{pop_volume_dir}/{pop_filename}"

existing_files = list_volume_files(pop_volume_dir)

if pop_filename in existing_files:
    print(f"  SKIP  {pop_filename} (already exists)")
else:
    resp = requests.get(POPULATION_API_URL, headers={"User-Agent": USER_AGENT}, timeout=30)
    resp.raise_for_status()
    pop_json = resp.json()
    
    json_str = json.dumps(pop_json, indent=2)
    dbutils.fs.put(pop_local_path, json_str, overwrite=True)
    print(f"  DOWNLOADED  {pop_filename}  ({len(json_str):,} bytes)")
    print(f"  Records: {len(pop_json.get('data', []))}")
    print(f"  Source: {pop_json.get('annotations', {}).get('source_name', 'unknown')}")

print()

Step 3: Downloading Population API data...
  URL: https://honolulu-api.datausa.io/tesseract/data.jsonrecords?cube=acs_yg_total_population_1&drilldowns=Year%2CNation&locale=en&measures=Population

  SKIP  population.json (already exists)



In [0]:
# --- Step 4: Write Ingestion Metadata Manifest -------------------------------

print("=" * 70)
print("Step 4: Writing ingestion metadata manifest...")

manifest = {
    "ingestion_timestamp": datetime.now(timezone.utc).isoformat(),
    "bls_source_url": BLS_BASE_URL,
    "population_api_url": POPULATION_API_URL,
    "bls_files": sorted(bls_files),
    "bls_file_count": len(bls_files),
    "user_agent": USER_AGENT,
    "volume_path": RAW_VOLUME_PATH,
}

manifest_path = f"{RAW_VOLUME_PATH}/ingestion_manifest.json"
manifest_str = json.dumps(manifest, indent=2)
dbutils.fs.put(manifest_path, manifest_str, overwrite=True)
print(f"  Manifest written to: {manifest_path}")
print()
print("=" * 70)
print("Ingestion complete!")
print(f"  BLS files: {len(bls_files)}")
print(f"  Volume: {RAW_VOLUME_PATH}")
print(f"  Manifest: {manifest_path}")

Step 4: Writing ingestion metadata manifest...
Wrote 729 bytes.
  Manifest written to: /Volumes/rearc_quest/bronze/raw_data/ingestion_manifest.json

Ingestion complete!
  BLS files: 12
  Volume: /Volumes/rearc_quest/bronze/raw_data
  Manifest: /Volumes/rearc_quest/bronze/raw_data/ingestion_manifest.json


In [0]:
# --- Step 5: Verify Volume Contents ------------------------------------------

print("=" * 70)
print("Step 5: Verifying Volume contents...")
print()

for subdir in [BLS_SUBDIR, POP_SUBDIR]:
    dir_path = f"{RAW_VOLUME_PATH}/{subdir}"
    print(f"{dir_path}/")
    try:
        files = dbutils.fs.ls(dir_path)
        for f in files:
            print(f"  {f.name:40s}  {f.size:>12,} bytes")
    except Exception as e:
        print(f"  Error: {e}")
    print()

print(f"{RAW_VOLUME_PATH}/")
try:
    files = dbutils.fs.ls(RAW_VOLUME_PATH)
    for f in files:
        print(f"  {f.name:40s}  {f.size:>12,} bytes")
except Exception as e:
    print(f"  Error: {e}")

Step 5: Verifying Volume contents...

/Volumes/rearc_quest/bronze/raw_data/bls_pr/
  pr.class                                           102 bytes
  pr.contacts                                        562 bytes
  pr.data.0.Current                            1,615,931 bytes
  pr.data.1.AllData                            3,239,525 bytes
  pr.duration                                        176 bytes
  pr.footnote                                         40 bytes
  pr.measure                                         745 bytes
  pr.period                                          146 bytes
  pr.seasonal                                         79 bytes
  pr.sector                                          263 bytes
  pr.series                                       15,657 bytes
  pr.txt                                          18,343 bytes

/Volumes/rearc_quest/bronze/raw_data/population/
  population.json                                  1,972 bytes

/Volumes/rearc_quest/bronze/raw_data/
  bls_pr/